# Functional Programming

Python supports first-class functions: functions are objects you can pass, return, and store. The `functools` module builds on this with memoisation, partial application, and utility decorators. Pair these with `map`, `filter`, and `sorted`'s `key=` argument and you get expressive, composable code with no loops in sight.

**What's inside:** `map`, `filter`, `sorted` with `key=`, `functools.reduce`, `partial`, `lru_cache`/`cache`, `wraps`, `total_ordering`, and the `operator` module.

**Learn more:** [functools](https://docs.python.org/3/library/functools.html) · [operator](https://docs.python.org/3/library/operator.html)

## 1. map, filter, and sorted

### 1.1 map: transform every element

In [1]:
names = ['alice', 'bob', 'carol']
upper = list(map(str.upper, names))
print(upper)

# with a lambda
squares = list(map(lambda x: x**2, range(6)))
print(squares)

['ALICE', 'BOB', 'CAROL']
[0, 1, 4, 9, 16, 25]


### 1.2 filter: keep elements that satisfy a predicate

In [2]:
nums = range(-5, 6)
positives = list(filter(lambda x: x > 0, nums))
print(positives)

# filter(None, iterable) drops falsy values
mixed = [0, 1, '', 'hello', None, 42, [], [1]]
print(list(filter(None, mixed)))

[1, 2, 3, 4, 5]
[1, 'hello', 42, [1]]


### 1.3 sorted with key: sort by any attribute

In [3]:
words = ['banana', 'fig', 'apple', 'date', 'elderberry']

# sort by length, then alphabetically within equal lengths
print(sorted(words, key=lambda w: (len(w), w)))

# sort dicts by a field
people = [{'name': 'Alice', 'age': 32}, {'name': 'Bob', 'age': 25}, {'name': 'Carol', 'age': 28}]
print(sorted(people, key=lambda p: p['age']))

['fig', 'date', 'apple', 'banana', 'elderberry']
[{'name': 'Bob', 'age': 25}, {'name': 'Carol', 'age': 28}, {'name': 'Alice', 'age': 32}]


## 2. functools.reduce

In [4]:
from functools import reduce

# reduce folds a sequence into a single value
total = reduce(lambda acc, x: acc + x, range(1, 6))
print(total)   # 1+2+3+4+5 = 15

# with an initial value
product = reduce(lambda acc, x: acc * x, range(1, 6), 1)
print(product)  # 1*2*3*4*5 = 120

# flatten nested lists
nested = [[1, 2], [3, 4], [5]]
flat = reduce(lambda a, b: a + b, nested)
print(flat)

15
120
[1, 2, 3, 4, 5]


## 3. functools.partial: freeze arguments

In [5]:
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube   = partial(power, exp=3)

print(square(5))   # 25
print(cube(3))     # 27
print(list(map(square, range(6))))

25
27
[0, 1, 4, 9, 16, 25]


In [6]:
from functools import partial

# partial is useful with sorted, map, filter
def starts_with(prefix, word):
    return word.startswith(prefix)

words = ['apple', 'apricot', 'banana', 'avocado', 'blueberry']
starts_a = partial(starts_with, 'a')
print(list(filter(starts_a, words)))

['apple', 'apricot', 'avocado']


## 4. functools.lru_cache and cache: memoisation

In [7]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(50))              # instant, despite 50 recursive calls
print(fib.cache_info())     # hits, misses, maxsize, currsize

12586269025
CacheInfo(hits=48, misses=51, maxsize=None, currsize=51)


In [8]:
from functools import cache  # Python 3.9+: unbounded lru_cache

@cache
def count_paths(m, n):
    """Unique paths in an m×n grid moving only right or down."""
    if m == 1 or n == 1:
        return 1
    return count_paths(m - 1, n) + count_paths(m, n - 1)

print(count_paths(10, 10))  # 48620

48620


## 5. functools.wraps: preserve function metadata

In [9]:
from functools import wraps

def log_calls(func):
    @wraps(func)           # copies __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        print(f'calling {func.__name__}')
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b):
    """Return a + b."""
    return a + b

print(add(2, 3))
print(add.__name__)    # 'add', not 'wrapper'
print(add.__doc__)     # 'Return a + b.'

calling add
5
add
Return a + b.


## 6. functools.total_ordering: derive comparison methods

In [10]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major, minor, patch):
        self.v = (major, minor, patch)

    def __eq__(self, other):
        return self.v == other.v

    def __lt__(self, other):     # total_ordering derives <=, >, >= from __eq__ and __lt__
        return self.v < other.v

    def __repr__(self):
        return '.'.join(map(str, self.v))

versions = [Version(1, 10, 0), Version(2, 0, 1), Version(1, 9, 3)]
print(sorted(versions))          # [1.9.3, 1.10.0, 2.0.1]
print(max(versions))             # 2.0.1

[1.9.3, 1.10.0, 2.0.1]
2.0.1


## 7. The operator module: functions for operators

In [11]:
import operator
from functools import reduce

# operator replaces lambdas for common operations
nums = [3, 1, 4, 1, 5, 9, 2, 6]
print(sorted(nums, key=operator.neg))          # reverse sort without reverse=True
print(reduce(operator.mul, range(1, 6)))       # 120  (5!)

# attrgetter / itemgetter for sorting by attribute/key
from operator import itemgetter, attrgetter

records = [('Alice', 32), ('Bob', 25), ('Carol', 28)]
print(sorted(records, key=itemgetter(1)))      # sort by age

[9, 6, 5, 4, 3, 2, 1, 1]
120
[('Bob', 25), ('Carol', 28), ('Alice', 32)]
